# Week 1 — Session 2 (Wednesday): The Hill-Climbing Intuition & Autograd Lab

**Instructors:** MSc. Antonio Aguilar (IMC, Pontifical Catholic University of Chile) & Dr. Luis Aguilar Ibáñez (National University of Piura, Perú)  
**Course:** Deep Learning: Foundations, Systems & Scientific AI Auditing  
**Environment:** Local Workstation (Miniconda + VS Code) or Google Colab

---

## 🎯 Lab Objectives
In this hands-on lab, we bring our tensors to life by mastering the engine of modern Artificial Intelligence: **Automatic Differentiation (Autograd)** and **Gradient Descent**.

By the end of this session, you will be able to:
1. **Enable gradient tracking** on model parameters using `requires_grad=True`.
2. **Execute the Backward Pass** with `loss.backward()` to compute sensitivities automatically.
3. **Understand the "Knob Sensitivity" intuition** in live Python code.
4. **Master the 2 Golden Rules of PyTorch**: Resetting accumulated gradients (`w.grad = None`) and updating weights safely inside `with torch.no_grad():`.
5. **Build and train your first Machine Learning model from scratch**: Fitting a line ($y = w \cdot x + b$) to noisy data using pure Gradient Descent without any library shortcuts.
6. **Visualize the learning process**: Plotting the loss curve dropping over time and watching your line lock onto the data!

---

## 0. Environment Setup & Imports

Let's import our numerical libraries and set random seeds so everyone gets consistent, reproducible results.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"✅ PyTorch Version: {torch.__version__}")
print(f"✅ NumPy Version:   {np.__version__}")
print(f"💻 Device:          {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU (Local)'}")

---
## 1. PyTorch's Secret Superpower: `requires_grad=True` & `loss.backward()`

In Deep Learning, we have two types of data:
1. **Input Data ($\mathbf{x}$) and Target Labels ($\mathbf{y}$):** These are fixed measurements from your dataset. They **do not** change during training (`requires_grad=False`).
2. **Model Parameters (Weights $w$, Biases $b$):** These are the internal adjustable dials that the computer tunes to make better predictions. They **must** track gradients (`requires_grad=True`).

Let's see this in action with a simple calculation!
$$
\large x = 2, y = 10
$$

$$
\large w=3, b = 1
$$

$$
\large \hat{y} = w \cdot x + b
$$

In [ ]:
# 1. Create learnable parameters with gradient tracking enabled
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# 2. Input data (Fixed measurements - no gradients needed)
x = torch.tensor(2.0)
target = torch.tensor(10.0)

# 3. Forward Pass: compute prediction and error (loss)
# Prediction: y = 3.0 * 2.0 + 1.0 = 7.0
y_hat = w * x + b

# Loss: Squared difference between prediction (7.0) and target (10.0)
# Loss: (7.0 - 10.0)^2 = (-3.0)^2 = 9.0
loss = (y_hat - target) ** 2

print(f"Prediction: {y_hat.item():.2f}")
print(f"Loss:       {loss.item():.2f}")

### 1.1 The Magic Line: `loss.backward()`

Before calling `.backward()`, `w.grad` is empty (`None`).

In [ ]:
print(f"Before backward -> w.grad: {w.grad}")
print(f"Before backward -> b.grad: {b.grad}")

When we call `loss.backward()`, PyTorch walks backwards through the mathematical operations and calculates the exact gradient (sensitivity) for every tensor that has `requires_grad=True`!

In [ ]:
# Trigger the backward pass!
loss.backward()

print(f"After backward  -> w.grad: {w.grad}")
print(f"After backward  -> b.grad: {b.grad}")

### 1.2 Visualizing the Unfolded Computational Graph (Top-Down Flow)

Behind the scenes, PyTorch connects your parameters ($w, b$), inputs ($x$), and targets ($y$) into a complete **Directed Acyclic Graph (DAG)** of mathematical operations.

#### 🧮 Mathematical Model of the Single Neuron:
1. **Linear Pre-activation ($z$):**
   $$\large u = w \cdot x = 3.0 \times 2.0 = 6.0$$
   $$\large z = u + b = 6.0 + 1.0 = 7.0$$
2. **Activation Output ($\hat{y}$):**
   $$\large \hat{y} = \sigma(z) = \text{Identity}(7.0) = 7.0$$
3. **Squared Error Loss ($\mathcal{L}$):**
   $$\large \text{error} = \hat{y} - y = 7.0 - 10.0 = -3.0$$
   $$\large \mathcal{L}(\hat{y}, y) = (\hat{y} - y)^2 = (-3.0)^2 = 9.0$$

Here is our helper function `show_autograd_graph` (from `autograd_viz.py`) that traces and **unfolds the entire graph vertically** (`flowchart TD`), showing how data and weights feed into arithmetic operations all the way to the final scalar loss:

In [ ]:
# Import our helper function
from autograd_viz import show_autograd_graph

# 1. Setup sample single-neuron forward pass
w_viz = torch.tensor(3.0, requires_grad=True)
b_viz = torch.tensor(1.0, requires_grad=True)
x_viz = torch.tensor(2.0)
y_viz = torch.tensor(10.0)

y_hat_viz = w_viz * x_viz + b_viz
loss_viz = (y_hat_viz - y_viz) ** 2

# 2. Render the complete unfolded computational graph!
show_autograd_graph(loss_viz, params={'w': w_viz, 'b': b_viz, 'x': x_viz, 'y': y_viz}, orientation="TD")

### ✏️ Exercise 1: Computing Sensitivities on a Quadratic Function

Let $y = x^2 + 3x + 5$.

Suppose we want to know how sensitive $y$ is to changes in $x$ when $x = 4.0$.

Complete the code below, call `.backward()`, and inspect `x.grad`.

In [ ]:
# TODO 1: Create a tensor x with value 4.0 and gradient tracking enabled
x = torch.tensor(4.0, requires_grad=True)

# TODO 2: Compute y = x^2 + 3*x + 5
y = ...

# TODO 3: Trigger the backward pass on y


print(f"Value of y at x=4: {y.item()}")
print(f"Sensitivity (x.grad): {x.grad.item()}")

# Verification: The derivative of x^2 + 3x + 5 is 2x + 3. At x=4, 2(4)+3 = 11.0!
assert x.grad.item() == 11.0, f"Expected gradient 11.0 but got {x.grad.item()}"
print("🎉 Fantastic! Exercise 1 passed successfully!")

In [ ]:
show_autograd_graph(loss_viz, params={'w': x, 'b': 5, 'x': x_viz, 'y': y_viz}, orientation="TD")

---
## 2. What Does the Gradient Actually Mean? (The "Knob" Experiment)

In our first example, we found:
- Current Weight: $w = 3.0$
- Current Loss: $\mathcal{L} = 9.0$
- Computed Gradient: $w.\text{grad} = -12.0$

In [ ]:
# Test the prediction with original w = 3.0
w_original = 3.0
loss_original = ((w_original * 2.0 + 1.0) - 10.0) ** 2
print(f"Loss with w = {w_original}:  {loss_original:.4f}")

What does $w.\text{grad} = -12.0$ tell us?

> **The Sensitivity Principle:** A **negative gradient** ($-12$) means that if we turn the knob $w$ to the **right** ($+\Delta w$), the loss will go **down**!

Let's test this directly by nudging $w$ slightly to the right ($w = 3.01$) and observing what happens to the loss!

In [ ]:
# de variation delta
delta_w = 0.01

In [ ]:
# Nudge w slightly to the right: w = 3.01 (increase of +0.01)
w_nudged_right = w_original + delta_w
loss_nudged_right = ((w_nudged_right * 2.0 + 1.0) - 10.0) ** 2
print(f"Loss with w = {w_nudged_right}:  {loss_nudged_right:.4f}")

And what about the left $(w = 2.99)$?

In [ ]:
# Nudge w slightly to the left: w = 3.01 (decrease of -0.01)
w_nudged_left = w_original - delta_w
loss_nudged_left = ((w_nudged_left * 2.0 + 1.0) - 10.0) ** 2
print(f"Loss with w = {w_nudged_left}:  {loss_nudged_left:.4f}")

So in summary:

In [ ]:
# Observe that the loss dropped!
delta_loss_right = loss_nudged_right - loss_original
delta_loss_left = loss_nudged_left - loss_original

print(f"Original Loss with w = {w_original}:  {loss_original:.4f}")
print(f"Loss with          w = {w_nudged_right}: {loss_nudged_right:.4f}  | Change in Loss: {delta_loss_right:.4f} (Loss went DOWN!)")
print(f"Loss with          w = {w_nudged_left}: {loss_nudged_left:.4f}  | Change in Loss: {delta_loss_left:.4f}  (Loss went UP!)")

---
## 3. The 2 Golden Rules of PyTorch Optimization

Before building a full training loop, we must understand two crucial rules in PyTorch.

### 3.1 Rule 1: Reset Gradients Every Step (`w.grad = None`)
PyTorch **accumulates (adds)** gradients into `.grad` with `+=` instead of overwriting them.

If you don't reset them between training steps, gradients from previous iterations will pile up and explode! Let's see this happen:

In [ ]:
w = torch.tensor(3.0, requires_grad=True)

# Step 1
loss1 = w * 2.0
loss1.backward()
print(f"Step 1 -> w.grad: {w.grad} (Correct gradient is 2.0)")

# Step 2 WITHOUT resetting gradients!
loss2 = w * 2.0
loss2.backward()
print(f"Step 2 -> w.grad: {w.grad} (Accumulated 2.0 + 2.0 = 4.0! ⚠️)")

# The Fix: Reset gradients before new computations!
w.grad = None
loss3 = w * 2.0
loss3.backward()
print(f"Step 3 with reset -> w.grad: {w.grad} (Clean gradient 2.0 ✅)")

### 3.2 Rule 2: Wrap Parameter Updates in `with torch.no_grad():`

When you update the parameters ($w \leftarrow w - \eta \cdot \text{gradient}$), you are adjusting the knobs, **not** adding a new mathematical layer to the model.

Wrapping the update in `with torch.no_grad():` tells PyTorch: *"Do not track this adjustment as part of the model flowchart!"*

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
lr = 0.05  # Learning rate

# Compute a dummy loss and backward
loss = (w * 2.0 + b - 10.0) ** 2
loss.backward()

# SAFE PARAMETER UPDATE:
with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad

# Reset gradients for next iteration
w.grad = None
b.grad = None

print(f"Updated w: {w.item():.4f}")
print(f"Updated b: {b.item():.4f}")

### ✏️ Exercise 2: Fix the Gradient Accumulation Bug!

The loop below was meant to take 3 optimization steps, but the programmer forgot to reset the gradients.

Add the missing line inside the loop so gradients don't accumulate!

In [ ]:
w = torch.tensor(5.0, requires_grad=True)
lr = 0.1

for step in range(3):
    loss = (w - 2.0) ** 2
    loss.backward()
    
    with torch.no_grad():
        w -= lr * w.grad
        
    # TODO: Add the line to reset w.grad here!
    w.grad = ...
    
    print(f"Step {step+1}: Loss = {loss.item():.4f}, New w = {w.item():.4f}")

# Verification check: With proper resets, w should smoothly approach 2.0
assert w.item() < 3.6 and w.item() > 2.0, "Check your gradient reset!"
print("🎉 Great job! Exercise 2 passed!")

---
## 4. Fitting a Line from Scratch ($y = w \cdot x + b$)

Now let's assemble everything into a complete Machine Learning training loop!

### 4.1 Generating Synthetic Data
Let's generate 100 noisy data points where the true relationship is:
$$\large y = 2.5 \cdot x + 1.0 + \text{noise}$$

In [ ]:
# Generate 100 random x values between -2 and 2
x = torch.linspace(-2, 2, 100)

# True parameters: w_true = 2.5, b_true = 1.0
true_w = 2.5
true_b = 1.0
noise = torch.randn(100) * 0.4  # Add random Gaussian noise

y_true = true_w * x + true_b + noise

# Plot the noisy data points
plt.figure(figsize=(7, 4))
plt.scatter(x.numpy(), y_true.numpy(), color="#38bdf8", alpha=0.7, label="Observed Data Points")
plt.title("Synthetic Data: Discovering the Underlying Relationship", fontsize=12)
plt.xlabel("Input Feature (x)")
plt.ylabel("Target Value (y)")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

### 4.2 The 5-Step Training Loop in Action

We start the computer with completely random guesses:
- Initial Guess: $w = 0.0, b = 0.0$ (a flat horizontal line).
- $\eta=0.1$
- Epochs = 60

Watch how the model learns over 50 iterations!

In [ ]:
# 1. Initialize random parameter guesses
w = ...
b = ...

learning_rate = ...
num_epochs = ...
loss_history = []

for epoch in range(num_epochs):
    # Step 1: Forward Pass (Predict)
    y_pred = ...
    
    # Step 2: Compute Error (Mean Squared Error)
    loss = ...
    loss_history.append(loss.item())
    
    # Step 3: Backward Pass (Compute Gradients)
    
    
    # Step 4: Update Parameters Downhill
    with torch.no_grad():
        w = ...
        b = ...
        
    # Step 5: Reset Gradients for next round
    w.grad = ...
    b.grad = ...
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}/{num_epochs} | Loss: {loss.item():.4f} | w: {w.item():.3f}, b: {b.item():.3f}")

print(f"\n🎯 True Values:     w = {true_w:.3f}, b = {true_b:.3f}")
print(f"🤖 Learned Values:  w = {w.item():.3f}, b = {b.item():.3f}")

### 4.3 Visualizing the Learning Process

Let's plot:
1. **The Loss Curve:** Showing the prediction error dropping over time.
2. **The Fitted Line:** Showing how well our learned model matches the data points!

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Plot 1: Loss Curve
axes[0].plot(loss_history, color="#f43f5e", linewidth=2.5)
axes[0].set_title("Training Loss Over Time", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch (Training Step)")
axes[0].set_ylabel("Mean Squared Error")
axes[0].grid(True, alpha=0.25)

# Plot 2: Learned Line vs Data
axes[1].scatter(x.numpy(), y_true.numpy(), color="#38bdf8", alpha=0.6, label="Data Points")
axes[1].plot(x.numpy(), (w.item() * x.numpy() + b.item()), color="#10b981", linewidth=3.0, label=f"Learned Line: y = {w.item():.2f}x + {b.item():.2f}")
axes[1].set_title("Fitted Model vs Data", fontsize=12, fontweight="bold")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend()
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

---
## 5. Exploring the Learning Rate (The Footstep Size)

How does the learning rate ($\eta$) impact not just the **loss curve**, but the **actual movement of the prediction line ($\hat{y} = w \cdot x + b$)** across the data?

Let's compare the 3 classic learning rate regimes:
1. **🐌 Too Small ($lr = 0.005$):** The line takes microscopic steps. Even after 40 epochs, the line is stuck near the bottom (underfitting).
2. **🎯 Just Right ($lr = 0.1$):** The line smoothly pivots and translates, locking onto the noisy data points within 40 epochs.
3. **💥 Too Large ($lr = 1.8$):** The steps overshoot the valley so violently that the prediction line swings wildly with extreme alternating slopes until the parameters explode!

In [ ]:
# Import learning rate experiment dashboard helper
from autograd_viz import plot_learning_rate_experiments

# Compare the 3 learning rate regimes across 40 epochs
plot_learning_rate_experiments(x, y_true, learning_rates=[0.005, 0.1, 1.8], epochs=40)

---
## 6. ✏️ Exercise 3: Hands-On Trend Discovery (Study Time vs. Performance Score)

Suppose an academic department collects experimental data measuring how **preparation / practice time ($x$, in hours)** impacts an **evaluation score ($y$, in points)**. You are provided with 50 student observations below:

In [ ]:
# 50 student study hour observations between 0 and 5 hours
samples = 50
study_hours = torch.linspace(0, 5, samples)
print(study_hours)

The underlying relationship has:
- A **baseline score** ($b$): The average score achieved with 0 hours of preparation ($b_{\text{true}} = 10.0$ points).
- A **rate of improvement** ($w$): The points gained per hour of practice ($w_{\text{true}} = 3.0$ points/hour).

$$\large \text{Score} = w \cdot \text{hours} + b + \text{noise}$$

In [ ]:
true_w = 3.0
true_b = 10.0
scores_measured = true_w * study_hours + true_b + torch.randn(samples) * 0.4

Your goal is to train a PyTorch model from scratch using the 5-step Gradient Descent loop to discover the optimal $w$ and $b$!

- Epochs = 200
- $lr=0.05$

In [ ]:
# TODO 1: Initialize w and b with requires_grad=True (start with initial guess 0.0)
w_model = ...
b_model = ...

learning_rate = ...
num_steps = ...

# TODO 2: Complete the 5-step training loop
for step in range(num_steps):
    # Step 1: Forward pass (compute predicted scores)
    scores_pred = ...
    
    # Step 2: Compute Mean Squared Error loss
    loss = ...
    
    # Step 3: Backward pass (compute gradients)
    
    
    # Step 4: Update parameters downhill
    with torch.no_grad():
        w_model = ...
        b_model = ...
        
    # Step 5: Reset gradients for next round
    w_model.grad = ...
    b_model.grad = ...

print(f"Estimated Improvement Rate (w): {w_model.item():.2f} pts/hr (True: {true_w:.2f})")
print(f"Estimated Baseline Score (b):   {b_model.item():.2f} pts    (True: {true_b:.2f})")

# Verification assertion
assert 2.7 < w_model.item() < 3.3, f"w should be close to 3.0, got {w_model.item():.2f}"
assert 9.5 < b_model.item() < 10.5, f"b should be close to 10.0, got {b_model.item():.2f}"
print("🎉 Outstanding! You trained an end-to-end Machine Learning model from raw data!")

---
## 7. Synthesis & Key Takeaways

Congratulations on completing Session 2! 🎓

Today, you mastered the core heartbeat of Artificial Intelligence:

| Concept | What It Does | Why It Matters |
| :--- | :--- | :--- |
| `requires_grad=True` | Tells PyTorch to record operations on a tensor | Designates parameters as learnable |
| `loss.backward()` | Traverses the computation graph backwards | Calculates exact sensitivities ($
abla \mathcal{L}$) in 1 line |
| `w.grad` | Holds the computed derivative of loss w.r.t $w$ | Directs whether to increase or decrease $w$ |
| `w.grad = None` | Resets gradient storage to clean state | Prevents gradients from blowing up |
| `with torch.no_grad():` | Disables tracking during parameter updates | Keeps parameter adjustments out of model graph |
| **Gradient Descent** | $\theta \leftarrow \theta - \eta \nabla \mathcal{L}$ | Moves parameters downhill toward minimum error |

---

## 🚀 Coming Up on Friday (Session 3)
In our next session, we make the leap **From Lines to Multi-Layer Neural Networks**:
- **Why Linear Models Are Not Enough:** The need for non-linear Activation Functions (Sigmoid, Tanh, ReLU).
- **The Artificial Perceptron:** How inputs, weights, biases, and activations form a neuron.
- **The PyTorch Blueprint:** Building reusable neural network architectures with `torch.nn.Module`.

See you on Friday!